## Imports

In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, roc_curve

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import tensorflow as tf
from tensorflow import keras
import pickle

## Accessing the data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
PATH = "/content/drive/MyDrive/academic_sucess_predictor/"
%cd  {PATH}

/content/drive/MyDrive/academic_sucess_predictor


In [12]:
scaler_filename = f"{PATH}scaler.pkl"
with open(scaler_filename, 'rb') as file:
    scaler = pickle.load(file)

print(f"StandardScaler loaded successfully from: {scaler_filename}")

StandardScaler loaded successfully from: /content/drive/MyDrive/academic_sucess_predictor/scaler.pkl


## Model Selection

In [ ]:
def load_model_from_drive(model_filename):
  """Loads a Keras model from Google Drive and returns it."""
  global PATH
  model_load_path = f"{PATH}{model_filename}.keras"

  try:
    loaded_model = keras.models.load_model(model_load_path)
    print(f"Model '{model_filename}' loaded successfully from: {model_load_path}")
    return loaded_model
  except Exception as e:
    print(f"Error loading model '{model_filename}': {e}")
    return None

In [ ]:
loaded_model = load_model_from_drive("LRModel")

loaded_model.summary()

Model 'LRModel' loaded successfully from: /content/drive/MyDrive/academic_sucess_predictor/LRModel.keras


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 1)              │            18 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 56 (228.00 B)

 Trainable params: 18 (72.00 B)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 38 (156.00 B)

## Prediction Model Functions

In [13]:
# Columns used by the model to predict
model_columns = [
    'Marital status',
    'Daytime/evening attendance',
    'Previous qualification',
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
    'Displaced',
    'Educational special needs',
    'Debtor',
    'Tuition fees up to date',
    'Gender',
    'Scholarship holder',
    'International',
    'Previous qualification (grade)',
    'Admission grade',
    'Age at enrollment'
]

options = {
    'Marital status': {
        'Single': 1,
        'Married': 2,
        'Widower': 3,
        'Divorced': 4,
        'Facto union': 5,
        'Legally separated': 6
    },
    'Daytime/evening attendance': {
        'Evening': 0,
        'Daytime': 1
    },
    'Displaced': {
        'No': 0,
        'Yes': 1
    },
    'Educational special needs': {
        'No': 0,
        'Yes': 1
    },
    'Debtor': {
        'No': 0,
        'Yes': 1
    },
    'Tuition fees up to date': {
        'No': 0,
        'Yes': 1
    },
    'Gender': {
        'Female': 0,
        'Male': 1
    },
    'Scholarship holder': {
        'No': 0,
        'Yes': 1
    },
    'International': {
        'No': 0,
        'Yes': 1
    }
}

previous_qualification_options = {
    "Secondary education": 1,
    "Higher education - bachelor's degree": 2,
    "Higher education - degree": 3,
    "Higher education - master's": 4,
    "Higher education - doctorate": 5,
    "Frequency of higher education": 6,
    "12th year of schooling - not completed": 9,
    "11th year of schooling - not completed": 10,
    "Other - 11th year of schooling": 12,
    "10th year of schooling": 14,
    "10th year of schooling - not completed": 15,
    "Basic education 3rd cycle (9th/10th/11th year) or equiv.": 19,
    "Basic education 2nd cycle (6th/7th/8th year) or equiv.": 38,
    "Technological specialization course": 39,
    "Higher education - degree (1st cycle)": 40,
    "Professional higher technical course": 42,
    "Higher education - master (2nd cycle)": 43
}

mother_qualification_options = {
    "Secondary Education - 12th Year of Schooling or Eq.": 1,
    "Higher Education - Bachelor's Degree": 2,
    "Higher Education - Degree": 3,
    "Higher Education - Master's": 4,
    "Higher Education - Doctorate": 5,
    "Frequency of Higher Education": 6,
    "12th Year of Schooling - Not Completed": 9,
    "11th Year of Schooling - Not Completed": 10,
    "7th Year (Old)": 11,
    "Other - 11th Year of Schooling": 12,
    "10th Year of Schooling": 14,
    "General commerce course": 18,
    "Basic Education 3rd Cycle (9th/10th/11th Year) or Equiv.": 19,
    "Technical-professional course": 22,
    "7th year of schooling": 26,
    "2nd cycle of the general high school course": 27,
    "9th Year of Schooling - Not Completed": 29,
    "8th year of schooling": 30,
    "Unknown": 34,
    "Can't read or write": 35,
    "Can read without having a 4th year of schooling": 36,
    "Basic education 1st cycle (4th/5th year) or equiv.": 37,
    "Basic Education 2nd Cycle (6th/7th/8th Year) or Equiv.": 38,
    "Technological specialization course": 39,
    "Higher education - degree (1st cycle)": 40,
    "Specialized higher studies course": 41,
    "Professional higher technical course": 42,
    "Higher Education - Master (2nd cycle)": 43,
    "Higher Education - Doctorate (3rd cycle)": 44
}

father_qualification_options = {
    "Secondary Education - 12th Year of Schooling or Eq.": 1,
    "Higher Education - Bachelor's Degree": 2,
    "Higher Education - Degree": 3,
    "Higher Education - Master's": 4,
    "Higher Education - Doctorate": 5,
    "Frequency of Higher Education": 6,
    "12th Year of Schooling - Not Completed": 9,
    "11th Year of Schooling - Not Completed": 10,
    "7th Year (Old)": 11,
    "Other - 11th Year of Schooling": 12,
    "2nd year complementary high school course": 13,
    "10th Year of Schooling": 14,
    "General commerce course": 18,
    "Basic Education 3rd Cycle (9th/10th/11th Year) or Equiv.": 19,
    "Complementary High School Course": 20,
    "Technical-professional course": 22,
    "Complementary High School Course - not concluded": 25,
    "7th year of schooling": 26,
    "2nd cycle of the general high school course": 27,
    "9th Year of Schooling - Not Completed": 29,
    "8th year of schooling": 30,
    "General Course of Administration and Commerce": 31,
    "Supplementary Accounting and Administration": 33,
    "Unknown": 34,
    "Can't read or write": 35,
    "Can read without having a 4th year of schooling": 36,
    "Basic education 1st cycle (4th/5th year) or equiv.": 37,
    "Basic Education 2nd Cycle (6th/7th/8th Year) or Equiv.": 38,
    "Technological specialization course": 39,
    "Higher education - degree (1st cycle)": 40,
    "Specialized higher studies course": 41,
    "Professional higher technical course": 42,
    "Higher Education - Master (2nd cycle)": 43,
    "Higher Education - Doctorate (3rd cycle)": 44
}

mother_occupation_options = {
    "Student": 0,
    "Representatives of the Legislative Power and Executive Bodies, Directors, Directors and Executive Managers": 1,
    "Specialists in Intellectual and Scientific Activities": 2,
    "Intermediate Level Technicians and Professions": 3,
    "Administrative staff": 4,
    "Personal Services, Security and Safety Workers and Sellers": 5,
    "Farmers and Skilled Workers in Agriculture, Fisheries and Forestry": 6,
    "Skilled Workers in Industry, Construction and Craftsmen": 7,
    "Installation and Machine Operators and Assembly Workers": 8,
    "Unskilled Workers": 9,
    "Armed Forces Professions": 10,
    "Other Situation": 90,
    "(blank)": 99,
    "Health professionals": 122,
    "Teachers": 123,
    "Specialists in information and communication technologies (ICT)": 125,
    "Intermediate level science and engineering technicians and professions": 131,
    "Technicians and professionals, of intermediate level of health": 132,
    "Intermediate level technicians from legal, social, sports, cultural and similar services": 134,
    "Office workers, secretaries in general and data processing operators": 141,
    "Data, accounting, statistical, financial services and registry-related operators": 143,
    "Other administrative support staff": 144,
    "Personal service workers": 151,
    "Sellers": 152,
    "Personal care workers and the like": 153,
    "Skilled construction workers and the like, except electricians": 171,
    "Skilled workers in printing, precision instrument manufacturing, jewelers, artisans and the like": 173,
    "Workers in food processing, woodworking, clothing and other industries and crafts": 175,
    "Cleaning workers": 191,
    "Unskilled workers in agriculture, animal production, fisheries and forestry": 192,
    "Unskilled workers in extractive industry, construction, manufacturing and transport": 193,
    "Meal preparation assistants": 194
}

father_occupation_options = {
    "Student": 0,
    "Representatives of the Legislative Power and Executive Bodies, Directors, Directors and Executive Managers": 1,
    "Specialists in Intellectual and Scientific Activities": 2,
    "Intermediate Level Technicians and Professions": 3,
    "Administrative staff": 4,
    "Personal Services, Security and Safety Workers and Sellers": 5,
    "Farmers and Skilled Workers in Agriculture, Fisheries and Forestry": 6,
    "Skilled Workers in Industry, Construction and Craftsmen": 7,
    "Installation and Machine Operators and Assembly Workers": 8,
    "Unskilled Workers": 9,
    "Armed Forces Professions": 10,
    "Other Situation": 90,
    "(blank)": 99,
    "Armed Forces Officers": 101,
    "Armed Forces Sergeants": 102,
    "Other Armed Forces personnel": 103,
    "Directors of administrative and commercial services": 112,
    "Hotel, catering, trade and other services directors": 114,
    "Specialists in the physical sciences, mathematics, engineering and related techniques": 121,
    "Health professionals": 122,
    "Teachers": 123,
    "Specialists in finance, accounting, administrative organization, public and commercial relations": 124,
    "Intermediate level science and engineering technicians and professions": 131,
    "Technicians and professionals, of intermediate level of health": 132,
    "Intermediate level technicians from legal, social, sports, cultural and similar services": 134,
    "Information and communication technology technicians": 135,
    "Office workers, secretaries in general and data processing operators": 141,
    "Data, accounting, statistical, financial services and registry-related operators": 143,
    "Other administrative support staff": 144,
    "Personal service workers": 151,
    "Sellers": 152,
    "Personal care workers and the like": 153,
    "Protection and security services personnel": 154,
    "Market-oriented farmers and skilled agricultural and animal production workers": 161,
    "Farmers, livestock keepers, fishermen, hunters and gatherers, subsistence": 163,
    "Skilled construction workers and the like, except electricians": 171,
    "Skilled workers in metallurgy, metalworking and similar": 172,
    "Skilled workers in electricity and electronics": 174,
    "Workers in food processing, woodworking, clothing and other industries and crafts": 175,
    "Fixed plant and machine operators": 181,
    "Assembly workers": 182,
    "Vehicle drivers and mobile equipment operators": 183,
    "Unskilled workers in agriculture, animal production, fisheries and forestry": 192,
    "Unskilled workers in extractive industry, construction, manufacturing and transport": 193,
    "Meal preparation assistants": 194,
    "Street vendors except food and street service providers": 195
}

options['Previous qualification'] = previous_qualification_options
options["Mother's qualification"] = mother_qualification_options
options["Father's qualification"] = father_qualification_options
options["Mother's occupation"] = mother_occupation_options
options["Father's occupation"] = father_occupation_options

In [14]:
# Widgets
inputs = {}

for col in model_columns:
    if col in options:
        inputs[col] = widgets.Dropdown(
            options=options[col],
            description=col,
            style={'description_width': '220px'},
            layout=widgets.Layout(width='600px')
        )
    elif col in ['Previous qualification (grade)', 'Admission grade']:
        inputs[col] = widgets.FloatSlider(
            value=120,
            min=0,
            max=200,
            step=1,
            description=col,
            style={'description_width': '220px'},
            layout=widgets.Layout(width='600px')
        )
    elif col == 'Age at enrollment':
        inputs[col] = widgets.IntSlider(
            value=20,
            min=17,
            max=70,
            step=1,
            description=col,
            style={'description_width': '220px'},
            layout=widgets.Layout(width='600px')
        )

In [15]:
def predict_student(button):
    with output:
        clear_output()

        student_data = {col: inputs[col].value for col in model_columns}
        student_df = pd.DataFrame([student_data])

        # Define continuous and categorical features based on the original scaling
        continuous_features = [
            'Previous qualification (grade)',
            'Admission grade',
            'Age at enrollment'
        ]
        categorical_features = [col for col in model_columns if col not in continuous_features]

        # Separate continuous and categorical features
        student_continuous = student_df[continuous_features]
        student_categorical = student_df[categorical_features]

        # Scale only the continuous features
        student_continuous_scaled = scaler.transform(student_continuous)

        # Convert scaled continuous features back to DataFrame
        student_continuous_scaled_df = pd.DataFrame(student_continuous_scaled, columns=continuous_features, index=student_df.index)

        # Concatenate scaled continuous features with unscaled categorical features
        # Ensure the order of columns is consistent with the original X
        student_final_processed_df = pd.concat([
            student_continuous_scaled_df,
            student_categorical
        ], axis=1)[model_columns] # Reorder columns to match the model's expected input

        probability = loaded_model.predict(student_final_processed_df)[0][0]

        threshold = 0.35 # Based on the model training and evaluation
        prediction = 'Dropout risk' if probability >= threshold else 'Graduate'

        print(f"Dropout probability: {probability:.2%}")
        print(f"Prediction using threshold {threshold}: {prediction}")

In [16]:
predict_button = widgets.Button(description='Predict dropout risk', button_style='primary')
output = widgets.Output()

predict_button.on_click(predict_student)

display(*inputs.values(), predict_button, output)

Dropdown(description='Marital status', layout=Layout(width='600px'), options={'Single': 1, 'Married': 2, 'Wido…

Dropdown(description='Daytime/evening attendance', layout=Layout(width='600px'), options={'Evening': 0, 'Dayti…

Dropdown(description='Previous qualification', layout=Layout(width='600px'), options={'Secondary education': 1…

Dropdown(description="Mother's qualification", layout=Layout(width='600px'), options={'Secondary Education - 1…

Dropdown(description="Father's qualification", layout=Layout(width='600px'), options={'Secondary Education - 1…

...

FloatSlider(value=120.0, description='Previous qualification (grade)', layout=Layout(width='600px'), max=200.0…

FloatSlider(value=120.0, description='Admission grade', layout=Layout(width='600px'), max=200.0, step=1.0, sty…

IntSlider(value=20, description='Age at enrollment', layout=Layout(width='600px'), max=70, min=17, style=Slide…

Button(button_style='primary', description='Predict dropout risk', style=ButtonStyle())

Output()